In [5]:
#importing packages
import json
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score, roc_curve)
tf.random.set_seed(42)
np.random.seed(42)


In [6]:
train_df=pd.read_csv('malaria_train.csv')
test_df=pd.read_csv('malaria_test.csv')
val_df=pd.read_csv('malaria_val.csv')

X_train=train_df.drop(columns=['Malaria_Positive']).values.astype(np.float32)
y_train = train_df['Malaria_Positive'].values.astype(np.float32)

X_val=val_df.drop(columns=['Malaria_Positive']).values.astype(np.float32)
y_val=val_df['Malaria_Positive'].values.astype(np.float32)


X_test=test_df.drop(columns=['Malaria_Positive']).values.astype(np.float32)
y_test=test_df['Malaria_Positive'].values.astype(np.float32)

print(f"Train: {X_train.shape}")
print(f"Test: {X_test.shape}")
print(f"Val: {X_val.shape}")
print(f"num of features: {X_train.shape[1]}")

FileNotFoundError: [Errno 2] No such file or directory: 'malaria_train.csv'

In [ ]:
import json
import numpy as np

with open("malaria_train.json") as f:
    metadata = json.load(f)

# metadata is a list of records
records = metadata

# Get feature columns, excluding the target
feature_cols = [
    key for key in records[0].keys()
    if key != "Malaria_Positive"
]

# Convert feature values into NumPy arrays
X = np.array(
    [[record[col] for col in feature_cols] for record in records],
    dtype=np.float32
)

# Get target values
y = np.array(
    [record["Malaria_Positive"] for record in records],
    dtype=np.float32
)

print(f"Loaded {len(feature_cols)} features")
print("Feature columns:", feature_cols)
print("X shape:", X.shape)
print("y shape:", y.shape)

In [ ]:
def build_ann(n_features):
    inputs = keras.Input(shape=(n_features,), name='features')

    #block1
    x=keras.layers.Dense(128, kernel_regularizer=keras.regularizers.l2(1e-4))(inputs)
    x=keras.layers.BatchNormalization()(x)
    x=keras.layers.Activation('relu')(x)
    x=keras.layers.Dropout(0.3)(x)


    #block 2
    x=keras.layers.Dense(64, kernel_regularizer=keras.regularizers.l2(1e-4))(x)
    x=keras.layers.BatchNormalization()(x)
    x=keras.layers.Activation('relu')(x)
    x=keras.layers.Dropout(0.3)(x)

     #block 3
    x=keras.layers.Dense(32)(x)
    x=keras.layers.Activation('relu')(x)
    x=keras.layers.Dropout(0.2)(x)

    outputs = keras.layers.Dense(1, activation='sigmoid', name='prob')(x)

    return keras.Model(inputs, outputs, name='MalariaANN')
    
model = build_ann(X_train.shape[1])
model.summary()



In [ ]:
model.compile(
    optimizer = keras.optimizers.Adam(learning_rate=5e-4),
    loss='binary_crossentropy',
    metrics=[
        'accuracy',
        keras.metrics.AUC(name='auc'),
        keras.metrics.Precision(name='precision'),
        keras.metrics.Recall(name='recall'),
    ]
)
  

In [ ]:
callbacks=[
    keras.callbacks.EarlyStopping(
        monitor='val_auc', patience=15,
        restore_best_weights=True, mode='max', verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_auc', factor=0.5, patience=10,
        min_lr=1e-6, mode='max', verbose=1
    ),
    keras.callbacks.ModelCheckpoint(
        'best_malaria.keras', monitor='val_auc',
        save_best_only=True, mode='max', verbose=0
    ),
]

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    batch_size=32,
    epochs=300,
    verbose=1,
    callbacks=callbacks
)
print(f"\nEpochs run:  {len(history.history['loss'])}")
print(f"Best val AUC:  {max(history.history['val_auc']):.4f}")


In [ ]:
test_loss, test_accuracy, test_auc, test_precision, test_recall = model.evaluate(
    X_test,
    y_test,
    verbose=1
)

print("Test Accuracy:", test_accuracy)
print("Test AUC:", test_auc)
print("Test Precision:", test_precision)
print("Test Recall:", test_recall)

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(history.history['loss'], label='Train')
axes[0].plot(history.history['val_loss'], label='Validation')
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(history.history['accuracy'], label='Train')
axes[1].plot(history.history['val_accuracy'], label='Validation')
axes[1].set_title('Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].legend()

axes[2].plot(history.history['auc'], label='Train')
axes[2].plot(history.history['val_auc'], label='validation')
axes[2].set_xlabel('Epoch')
axes[2].set_title('AUC')
axes[2].legend()

plt.suptitle('Training History', fontsize=13)
plt.tight_layout()
plt.savefig('training_history.png' , dpi=150, bbox_inches='tight')
plt.show

In [ ]:
print(history.history.keys())

In [ ]:
model.load_weights('best_malaria.keras')
test_results=model.evaluate(X_test, y_test, verbose=0)
for name, value in zip(model.metrics_names, test_results):
    print (f"{name:<12}: {value:.4f}")

In [ ]:
y_prob = model.predict(X_test, verbose=0).flatten()
auc_score = roc_auc_score(y_test, y_prob)

fpr, tpr, thresholds = roc_curve(y_test, y_prob)
plt.figure(figsize = (5,5))
plt.plot(fpr,tpr, color='crimson', label=f"ROC curve (AUC = {auc_score:.3f})")
plt.plot([0,1], [0,1], color = 'gray', linestyle='--', label='random guess')
plt.xlabel("False positive rate")
plt.ylabel("true positive rate")
plt.title("ROC curve")
plt.legend()
plt.tight_layout()
plt.savefig("roc_curve.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"ROC-AUC : {auc_score:.4f}")

In [ ]:
results = model.evaluate(X_test, y_test, verbose=1)

print("Test results:")
for name, value in zip(model.metrics_names, results):
    print(f"{name}: {value:.4f}")

In [ ]:
from sklearn.metrics import confusion_matrix

y_prob = model.predict(X_test)
y_pred = (y_prob >= 0.5).astype(int).ravel()

cm = confusion_matrix(y_test, y_pred)

print(cm)

In [ ]:
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

print("Training positives:", np.sum(y_train == 1))
print("Training negatives:", np.sum(y_train == 0))

print("Test positives:", np.sum(y_test == 1))
print("Test negatives:", np.sum(y_test == 0))
# print("x_features",X)

In [4]:
y_pred = (y_prob >= 0.5).astype(int)
import seaborn as sns
print(classification_report(y_test, y_pred, target_names=['NO malaria', 'malaria']))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(4.5, 4))
sns.heatmap(cm, annot=True, fmt='d' , cmap='Blues',
            xticklabels=['no_malaria', 'malaria'],
            yticklabels=['no_malaria', 'malaria'])
plt.ylabel('Actual')
plt.xlabel('predicted')
plt.title('Confusion Matrix')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()


NameError: name 'y_prob' is not defined

In [21]:
import numpy as np

feature_cols=[
    'Age',
    'Fever_Temp1',
    'Chills1',
    'Headache_Severity',
    'Vomiting',
   ' Diarrhea',
    'Muscle_Pain',
    'Fatigue_Level',
    'Sweating',
    'Nausea',
    'High_Fever',
    'Fever_x_Chills',
    'Severe_Headache',
    'Symptom_Count',
    'Vulnerable_Age'
]

def predict_malaria(patient, model):
    patient = pateient.copy()

    
   


